# S3 · 횡단 설문

이 노트북은 구글 **Colab**에서 바로 실행됩니다. 위에서부터 각 셀을 **Shift+Enter** 로 실행하세요. 설치는 없고, 구글 계정만 있으면 됩니다.

📖 본문 학습 페이지: [S3 · 횡단 설문](https://grow.minds.kr/textbooks/css-methods/causal/book/s3-시나리오-설문.html)

## 1. 준비

In [ ]:
# 이 책의 데이터·코드를 코랩으로 내려받습니다(처음 한 번, 수 초).
!git clone -q https://github.com/dataminds/css-methods-causal-code.git
%cd css-methods-causal-code

In [ ]:
import pandas as pd, numpy as np
from scipy import stats

def load(name, clean=True):
    df = pd.read_csv(f"data/journey_{name}.csv")
    return df[df.attn_1 == 1] if clean and "attn_1" in df else df

def ols(y, X):                      # 절편 포함 최소제곱 → (계수, 표준오차, p, R^2)
    y = np.asarray(y, float)
    X1 = np.column_stack([np.ones(len(y))] + [np.asarray(x, float) for x in X])
    b, *_ = np.linalg.lstsq(X1, y, rcond=None)
    resid = y - X1 @ b
    n, k = X1.shape
    se = np.sqrt(np.diag(resid @ resid / (n - k) * np.linalg.inv(X1.T @ X1)))
    p = 2 * stats.t.sf(np.abs(b / se), n - k)
    r2 = 1 - (resid @ resid) / ((y - y.mean()) @ (y - y.mean()))
    return b, se, p, r2

def cohen_d(a, b):
    sp = np.sqrt(((len(a)-1)*a.std(ddof=1)**2 + (len(b)-1)*b.std(ddof=1)**2) / (len(a)+len(b)-2))
    return (a.mean() - b.mean()) / sp

def cronbach(items):
    items = np.asarray(items, float); k = items.shape[1]
    return k/(k-1) * (1 - items.var(axis=0, ddof=1).sum() / items.sum(axis=1).var(ddof=1))

print("준비 끝. 데이터와 도우미 함수를 불러왔습니다.")


## 2. 위계 회귀: 무엇이 얼마를 더 설명하나
블록을 쌓으며 R² 증분을 본다. 여정이 .389, 조절 항이 .027 을 더한다.

In [ ]:
svy = load("svy")
s = svy[svy.gender.isin([1, 2])]                 # 성별 통제 분석은 554명
fem = (s.gender == 2).astype(float)
h = s.hjs - s.hjs.mean(); r = s.refl - s.refl.mean()
prev = 0
for lab, X in [("1 인구학", [s.age, fem]), ("2 + 여정", [s.age, fem, h]),
               ("3 + 조절", [s.age, fem, h, r, h*r])]:
    b, se, p, r2 = ols(s.mil, X)
    print(lab, round(r2, 4), " 증분", round(r2 - prev, 4)); prev = r2

## 3. 충돌 통제 함정
세대성(gen)을 통제하면 계수가 .982 에서 .865 로 **왜곡**된다. 통제가 언제나 좋은 것이 아니다(11장).

In [ ]:
b0, *_ = ols(s.mil, [s.hjs])
b1, *_ = ols(s.mil, [s.hjs, s.gen])
print("통제 없음:", round(b0[1], 3), " gen 통제:", round(b1[1], 3))    # 0.982 0.865

## 4. 방향은 자료가 안 정한다
의미를 여정으로 설명하나 여정을 의미로 설명하나 **R² 도 p 도 같다**.

In [ ]:
_, _, p1, r21 = ols(svy.mil, [svy.hjs])
_, _, p2, r22 = ols(svy.hjs, [svy.mil])
print(round(r21, 4), round(r22, 4))            # 0.3953 0.3953
print(f"{p1[1]:.1e}", f"{p2[1]:.1e}")          # 둘 다 1.3e-63

## 4. 직접 바꿔 보기
위 셀의 숫자(씨앗 73, 표본 크기, 제외 기준 등)를 바꿔 다시 실행해 보세요. 결과가 어떻게 달라지나요?

> **검증 로그(부록 B)**: 무엇을 바꿨고, 무엇이 나왔고, 예상과 같았는지 한 문단으로 적어 두세요. 실행이 아니라 검증이 이 책의 핵심입니다.